# Manifold-Matching Autoencoders — Colab quick start

This notebook clones the repo, installs dependencies, trains an MMAE,
and renders an animated GIF of the latent space evolving over epochs.

Tip: switch the runtime to GPU (Runtime → Change runtime type → GPU) for
faster training on MNIST. CPU works fine for `spheres` and `mammoth`.

## 1. Clone the repo and install dependencies

In [ ]:
import os, sys
REPO = 'manifold-matching-autoencoders'
if not os.path.exists(REPO):
    !git clone https://github.com/laurent-cheret/manifold-matching-autoencoders.git
%cd $REPO
!pip install -q -r requirements.txt

## 2. Pick what to train

Tweak `DATASET`, `REGULARIZER` (`mmae` or `none`), and `REFERENCE`
(`pca`, `umap`, `tsne`).

In [ ]:
DATASET     = 'mammoth'   # 'mnist' | 'fmnist' | 'spheres' | 'mammoth'
REGULARIZER = 'mmae'      # 'mmae' | 'none'
REFERENCE   = 'pca'       # 'pca' | 'umap' | 'tsne'
REF_DIM     = 2
LATENT_DIM  = 2           # try 3 for mammoth and a 3D GIF
EPOCHS      = 60
BATCH_SIZE  = 256
LR          = 1e-3
LAM         = 1.0
SEED        = 42

## 3. Train

In [ ]:
from mmae import train_run

result = train_run(
    dataset=DATASET,
    regularizer=REGULARIZER,
    reference=REFERENCE,
    ref_dim=REF_DIM,
    lam=LAM,
    latent_dim=LATENT_DIM,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    seed=SEED,
    snapshot_every=1,   # capture latent each epoch for the GIF
)

## 4. Plot the loss curves

In [ ]:
import matplotlib.pyplot as plt

h = result.history
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h['train_total'], label='train total')
ax.plot(h['val_total'], label='val total')
if REGULARIZER == 'mmae':
    ax.plot(h['train_recon'], label='train recon', linestyle='--')
    ax.plot(h['train_mm'], label='train mm', linestyle='--')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Final latent space

In [ ]:
from mmae.viz import plot_latents

_, z, y = result.snapshots[-1]
plot_latents(z, y, title=f'{DATASET} · {REGULARIZER} · {REFERENCE}{REF_DIM}')
plt.show()

## 6. Build the per-epoch GIF and display it

In [ ]:
from mmae.viz import make_latent_gif
from IPython.display import Image

gif_path = f'latent_evolution_{DATASET}_{REGULARIZER}_{REFERENCE}{REF_DIM}.gif'
make_latent_gif(
    result.snapshots, gif_path, fps=8,
    title_prefix=f'{DATASET} · {REGULARIZER} · {REFERENCE}{REF_DIM}',
)
Image(filename=gif_path)

## 7. (Bonus) Compare with and without the regularizer

Run the same training with `regularizer='none'` and look at the difference.

In [ ]:
vanilla = train_run(
    dataset=DATASET, regularizer='none',
    latent_dim=LATENT_DIM, epochs=EPOCHS,
    batch_size=BATCH_SIZE, lr=LR, seed=SEED,
    snapshot_every=1,
)
_, z_v, y_v = vanilla.snapshots[-1]
plot_latents(z_v, y_v, title=f'{DATASET} · vanilla AE')
plt.show()